## DistilBert

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling
from transformers import BertForMaskedLM, BertTokenizerFast, BertConfig
from transformers import pipeline

torch.manual_seed(42)

In [33]:
device = torch.device("cuda" if torch.cuda.is_available else "cpu")

bert_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

bert_model = BertForMaskedLM.from_pretrained("google-bert/bert-base-uncased")

bert_model = bert_model.to(device)

config = BertConfig(vocab_size = bert_tokenizer.vocab_size,
                    hidden_size = 128,
                    num_hidden_layers = 2,
                    num_attention_heads = 4,
                    intermediate_size = 512,
                    max_position_embeddings = 128,
                    devicee = "cuda"
                    )
distilbert_model = BertForMaskedLM(config)
distilbert_model = distilbert_model.to(device)
# distilbert_model.cls.predictions.transform.dense = nn.Linear(in_features=128, out_features=768, bias=True, device=device)
# distilbert_model.cls.predictions.transform.LayerNorm = nn.LayerNorm((768,), eps=1e-12, elementwise_affine=True, device=device)
# distilbert_model.cls.predictions.decoder = nn.Linear(in_features=768, out_features=30522, bias=True, device=device)
student_proj = nn.Linear(in_features=128, out_features=768, bias=True, device=device)

print(distilbert_model)

# Freeze teacher model
bert_model.eval()
for param in bert_model.parameters():
  param.requires_grad = False

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 128, padding_idx=0)
      (position_embeddings): Embedding(128, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-1): 2 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=128, out_features=128, bias=True)
              (key): Linear(in_features=128, out_features=128, bias=True)
              (value): Linear(in_features=128, out_features=128, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=128, out_features=128, bias=True)
              (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_aff

In [ ]:
pipeline("fill-mask", model=bert_model, tokenizer=bert_tokenizer)("The capital of [MASK] is Mexico city.")

In [ ]:
pipeline("fill-mask", model=distilbert_model, tokenizer=bert_tokenizer)("The capital of [MASK] is Mexico city.")

In [34]:
# Get the final hidden states before the classification head
capt = {}
def hook_bert(module, input, output):
    capt["b_ln"] = output
handle_bert = bert_model.cls.predictions.transform.LayerNorm.register_forward_hook(hook_bert)
def hook_distilbert(module, input, output):
    capt["db_ln"] = output
handle_distilbert = distilbert_model.cls.predictions.transform.LayerNorm.register_forward_hook(hook_distilbert)



In [ ]:
# Load dataset
mlm_dataset_raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")

def tokenize(example, tokenizer=bert_tokenizer):
  return tokenizer(example["text"], truncation=True, max_length=128, padding="max_length")
mlm_dataset = mlm_dataset_raw.map(tokenize, batched=True)

mlm_collator = DataCollatorForLanguageModeling(bert_tokenizer, mlm = True, mlm_probability = 0.15)


train_dl = DataLoader(
    mlm_dataset.remove_columns('text'),
    batch_size=32,
    collate_fn=mlm_collator,
    shuffle=True,
    pin_memory=True,
    num_workers=2,
    drop_last=True
)


In [ ]:
# Train model
optimizer = torch.optim.Adam( list(distilbert_model.parameters()) + list(student_proj.parameters()), lr=0.003)
criterion1 = nn.CosineEmbeddingLoss()
criterion2 = nn.CrossEntropyLoss(ignore_index=-100) # CrossEntropyLoss ya aplica internamente un LogSoftmax
n_epochs = 10
a, b, c = 5, 2, 1

for epoch in range(n_epochs):
  student_proj.train()
  distilbert_model.train()
  epoch_loss = 0
  for sample in train_dl:
    X = sample['input_ids'].to(device)
    y_real = sample['labels'].to(device)
    attention_mask = sample['attention_mask'].to(device)
    # Forward
    y_db = distilbert_model(X, attention_mask=attention_mask)
    with torch.no_grad():
      y_b = bert_model(X, attention_mask=attention_mask)
      mask = y_real != -100
    capt["db_ln"].to(device)
    capt["b_ln"].to(device)
    loss1 = c*criterion1(student_proj(capt["db_ln"][mask]), capt["b_ln"][mask], target=torch.ones(capt["b_ln"][mask].size(0), device=device))
    loss2 = a*criterion2(y_db.logits.transpose(1, 2), torch.argmax(y_b.logits, dim=-1))
    loss3 = b*criterion2(y_db.logits.transpose(1, 2), y_real)
    total_loss = loss1 + loss2 + loss3
    # Backward
    total_loss.backward() # calcular gradiente de parametros
    # GDS
    optimizer.step() # actualizar parametros
    optimizer.zero_grad() # reset the gradients
    # Add loss
    epoch_loss += total_loss.item()
  print("epoch: %s, train total loss: %s" %(epoch, epoch_loss))

handle_bert.remove()
handle_distilbert.remove()


In [31]:
pipeline("fill-mask", model=distilbert_model, tokenizer=bert_tokenizer)("The capital of [MASK] is Mexico city.")

[{'score': 0.04157177358865738,
  'token': 1996,
  'token_str': 'the',
  'sequence': 'the capital of the is mexico city.'},
 {'score': 0.03683724254369736,
  'token': 1010,
  'token_str': ',',
  'sequence': 'the capital of, is mexico city.'},
 {'score': 0.032615453004837036,
  'token': 1998,
  'token_str': 'and',
  'sequence': 'the capital of and is mexico city.'},
 {'score': 0.02907416597008705,
  'token': 1997,
  'token_str': 'of',
  'sequence': 'the capital of of is mexico city.'},
 {'score': 0.02118205651640892,
  'token': 1012,
  'token_str': '.',
  'sequence': 'the capital of. is mexico city.'}]